In [0]:
# Databricks notebook source
# =============================================================================
# SILVER LAYER : CustShipToAddr Transformation (UNITY CATALOG PROD STANDARD)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.functions import broadcast
import logging

# ──────────────────────────────────────────────────────────────────────────────
# 1. ENVIRONMENT CONFIGURATION & OPTIMIZATIONS
# ──────────────────────────────────────────────────────────────────────────────
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_custshiptoaddr")
print("⚪ INITIALIZING PROD SILVER PIPELINE : CUST SHIP TO ADDRESS")

CATALOG_NAME = "prx"
BRONZE_SCHEMA = "prx_bronze"
SILVER_SCHEMA = "prx_silver"

# Fully Qualified Table Names
BRONZE_TABLE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_addresses"
CUSTOMER_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customer"
SILVER_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.custshiptoaddr"
EXCEPTION_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.central_exceptions"

STORAGE_ACCOUNT = "stpraxaslakehouse"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/custshiptoaddr"
EXCEPTION_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/central_exceptions"

MERGE_KEYS = ["CustShipToAddrSrcId"]

spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")
executing_user = spark.sql("SELECT current_user()").collect()[0][0]

# ──────────────────────────────────────────────────────────────────────────────
# 2. UTILITIES
# ──────────────────────────────────────────────────────────────────────────────
def safe_str(c):
    return F.coalesce(F.col(c).cast(StringType()), F.lit(""))

def empty_str_to_null_ts(c):
    return F.when(
        F.col(c).isNull() | (F.trim(F.col(c)) == ""), None
    ).otherwise(F.col(c)).cast(TimestampType())

# ──────────────────────────────────────────────────────────────────────────────
# 3. READ, FILTER & JOIN
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Reading Bronze Data from Unity Catalog...")

bronze_df = spark.table(BRONZE_TABLE)
customer_df = spark.table(CUSTOMER_TABLE).select("CustSrcId", "CustKey", "CustContactSrcId")

filtered_df = bronze_df.filter(F.col("AccountIsSupplier").isNotNull())

joined_df = filtered_df.alias("a").join(
    broadcast(customer_df.alias("c")),
    F.col("a.Account") == F.col("c.CustSrcId"),
    "inner"
)

# ──────────────────────────────────────────────────────────────────────────────
# 4. TRANSFORMATIONS
# ──────────────────────────────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────────
# 4. TRANSFORMATIONS (RESTORED PARITY WITH BILL-TO)
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Applying Business Transformations...")

silver_df = joined_df.select(
    # Core Keys
    safe_str("a.ID").alias("CustShipToAddrSrcId"),
    safe_str("a.Account").alias("CustSrcId"),
    safe_str("a.Contact").alias("CustContactSrcId"),
    safe_str("c.CustKey").alias("CustKey"),
    
    # Warehouse
    safe_str("a.Warehouse").alias("WHSrcId"),
    safe_str("a.WarehouseCode").alias("WHKey"),
    safe_str("a.WarehouseDescription").alias("WHName"),
    
    # Customer Info
    safe_str("a.AccountName").alias("CustName"),
    safe_str("a.ContactName").alias("CustContactName"),
    
    # Address
    safe_str("a.AddressLine1").alias("AddrLn1"),
    safe_str("a.AddressLine2").alias("AddrLn2"),
    safe_str("a.AddressLine3").alias("AddrLn3"),
    safe_str("a.City").alias("City"),
    safe_str("a.State").alias("State"),
    safe_str("a.StateDescription").alias("StateName"),
    safe_str("a.Country").alias("Country"),
    safe_str("a.CountryName").alias("CountryName"),
    safe_str("a.Postcode").alias("ZipCd"),
    
    # Communication
    safe_str("a.Mailbox").alias("EmailAddr"),
    safe_str("a.Phone").alias("TelPhNo"),
    safe_str("a.Fax").alias("FaxNo"),
    
    # Additional
    safe_str("a.Notes").alias("ShipInstr"),
    F.lit("Addresses").alias("SrcTable"),
    safe_str("a.Main").alias("Main"),
    
    # Source System Dates & Lineage (🔥 FIXED: Copied from Bill-To for parity!)
    empty_str_to_null_ts("a.Created").alias("CreatedDt"),
    empty_str_to_null_ts("a.Modified").alias("UpdatedDt"),
    safe_str("a.Creator").alias("CreatedBySrcId"),
    safe_str("a.CreatorFullName").alias("CreatedBy"),
    safe_str("a.Modifier").alias("UpdatedBySrcId"),
    safe_str("a.ModifierFullName").alias("UpdatedBy"),
    
    # Enterprise Databricks Pipeline Audit
    F.current_timestamp().alias("SysCreatedDt"),
    F.current_timestamp().alias("SysUpdatedDt"),
    F.lit(executing_user).alias("SysUpdatedBy")
)

# ──────────────────────────────────────────────────────────────────────────────
# 5. DATA QUALITY & DEDUPLICATION
# ──────────────────────────────────────────────────────────────────────────────
logger.info("Running Data Quality Checks...")

dq_df = silver_df.withColumn(
    "DQ_Reason",
    F.array_remove(F.array(
        F.when(F.col("CustShipToAddrSrcId") == "", F.lit("MISSING_SHIP_ID")),
        F.when(F.col("CustSrcId") == "", F.lit("MISSING_CUST_ID"))
    ), None)
).withColumn(
    "DQ_Status",
    F.when(F.size("DQ_Reason") > 0, F.lit(2)).otherwise(F.lit(0))
)

window_dup = Window.partitionBy("CustShipToAddrSrcId").orderBy(F.col("UpdatedDt").desc_nulls_last())

dq_df = dq_df.withColumn(
    "rn", F.row_number().over(window_dup)
).withColumn(
    "DQ_Status",
    F.when(F.col("rn") > 1, F.lit(4)).otherwise(F.col("DQ_Status"))
).drop("rn")

dq_df.cache()

valid_df = dq_df.filter(F.col("DQ_Status") == 0).drop("DQ_Reason", "DQ_Status")
error_df = dq_df.filter(F.col("DQ_Status").isin(2, 4))

valid_count = valid_df.count()
error_count = error_df.count()

logger.info(f"✅ Valid Records: {valid_count:,}")
logger.info(f"❌ Error Records: {error_count:,}")

# ──────────────────────────────────────────────────────────────────────────────
# 6. CENTRAL EXCEPTION ROUTING
# ──────────────────────────────────────────────────────────────────────────────
if error_count > 0:
    logger.info("Routing Exceptions to Unity Catalog Quarantine...")
    exception_df = error_df.select(
        F.lit(SILVER_TABLE).alias("table_name"),
        F.col("CustShipToAddrSrcId").alias("record_key"),
        F.when(F.col("DQ_Status") == 4, F.lit("Duplicate Record"))
         .otherwise(F.concat(F.lit("Validation Failed: "), F.concat_ws(", ", F.col("DQ_Reason")))).alias("exception_details"),
        F.current_timestamp().alias("syscreateddt")
    )

    if spark.catalog.tableExists(EXCEPTION_TABLE):
        exception_df.write.format("delta").mode("append").saveAsTable(EXCEPTION_TABLE)
    else:
        exception_df.write.format("delta").mode("overwrite").option("path", EXCEPTION_PATH).saveAsTable(EXCEPTION_TABLE)

# ──────────────────────────────────────────────────────────────────────────────
# 7. MERGE INTO SILVER DELTA
# ──────────────────────────────────────────────────────────────────────────────
if valid_count > 0:
    logger.info("Executing MERGE INTO Silver Delta...")
    window_spec = Window.partitionBy(*MERGE_KEYS).orderBy(F.col("UpdatedDt").desc_nulls_last())
    final_df = valid_df.withColumn("rn", F.row_number().over(window_spec)).filter("rn = 1").drop("rn")

    if spark.catalog.tableExists(SILVER_TABLE):
        delta_tbl = DeltaTable.forName(spark, SILVER_TABLE)
        cond = " AND ".join([f"tgt.{k} = src.{k}" for k in MERGE_KEYS])
        
        (delta_tbl.alias("tgt")
         .merge(final_df.alias("src"), cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        final_df.write.format("delta").mode("overwrite").option("path", SILVER_PATH).saveAsTable(SILVER_TABLE)

dq_df.unpersist()
print("🎉 PROD SHIP TO ADDRESS PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM prx.prx_silver.custshiptoaddr LIMIT 10;

In [0]:
%sql
SELECT * FROM prx.prx_silver.custshiptoaddr LIMIT 10